# Healthcare Sector Portfolio Optimization

**Analyst:** Anurag Pokala  
**Date:** February 5, 2026

## Overview

This notebook implements three portfolio optimization techniques for the Healthcare sector sleeve:
1. **Mean-Variance (Markowitz) Optimization**
2. **Black-Litterman with Sentiment Analysis (Pure Sentiment)**
3. **CVaR (Conditional Value-at-Risk) Optimization**

**Note:** Unlike the Consumer sector, this Healthcare sector optimization uses ONLY sentiment analysis for Black-Litterman, as lead analyst views were not provided. The sentiment analysis uses FinBERT and financial news APIs to generate forward-looking views.

## Section 1: Setup and Configuration

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

# Import custom modules
from src import data_loader, estimators, constraints, mv_optimizer, bl_model, metrics, reporting, backtester, sentiment_analyzer, cvar_optimizer

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All libraries imported successfully")

In [ ]:
# Load Healthcare sector configuration
with open('../config_healthcare.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Extract key parameters
tickers = config['tickers']
before_weights = np.array([config['before_weights'][t] for t in tickers])
start_date = config['data']['start_date']
end_date = config['data']['end_date']
rf = config['risk_free_rate']

print("Healthcare Sector Configuration Loaded:")
print(f"  Tickers: {', '.join(tickers)}")
print(f"  Date range: {start_date} to {end_date}")
print(f"  Risk-free rate: {rf:.2%}")
print(f"  Portfolio value: ${config['portfolio']['total_value']:,}")
print(f"\nBefore Portfolio Weights:")
for ticker, weight in zip(tickers, before_weights):
    print(f"  {ticker}: {weight:.2%}")
print(f"  Total: {before_weights.sum():.2%}")
print(f"\nConstraints:")
print(f"  Long-only: {config['constraints']['long_only']}")
print(f"  Fully invested: {config['constraints']['fully_invested']}")
print(f"  Max weight: {config['constraints']['max_weight']:.0%}")

## Section 2: Data Loading and Preprocessing

Load 3 years of daily price data for Healthcare sector stocks.

In [ ]:
# Load price data
prices = data_loader.load_prices(tickers, start_date, end_date)

# Display summary
print("\nFirst 5 days:")
print(prices.head())
print("\nLast 5 days:")
print(prices.tail())

In [ ]:
# Compute returns
returns = data_loader.compute_returns(prices)

print("Returns Summary (simple returns):")
print(f"Shape: {returns.shape}")
print(f"Date range: {returns.index[0].date()} to {returns.index[-1].date()}")
print("\nDescriptive Statistics (annualized):")
stats = data_loader.get_summary_statistics(returns)
print(stats)

## Section 3: Covariance and Returns Estimation

Estimate expected returns and covariance matrix using robust methods.

In [ ]:
# Estimate expected returns (historical mean)
mu_hist = estimators.estimate_expected_returns(returns, shrinkage=0.0)

print("Expected Returns (annualized):")
print("="*50)
for ticker, ret in zip(tickers, mu_hist):
    print(f"{ticker:6s}: {ret:7.2%}")
print("="*50)

In [ ]:
# Estimate covariance matrix (Ledoit-Wolf shrinkage)
Sigma, shrinkage_intensity = estimators.estimate_covariance_shrinkage(returns)

print(f"Covariance Matrix Estimated (Ledoit-Wolf)")
print(f"Shrinkage intensity: {shrinkage_intensity:.4f}")
print(f"\nAnnualized Volatilities:")
print("="*50)
for ticker, vol in zip(tickers, np.sqrt(np.diag(Sigma.values))):
    print(f"{ticker:6s}: {vol:7.2%}")
print("="*50)

In [ ]:
# Correlation matrix heatmap
corr_matrix = returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix - Healthcare Sector', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
# Find highest correlations
corr_flat = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
print(f"  Average correlation: {corr_flat.mean():.3f}")
print(f"  Max correlation: {corr_flat.max():.3f}")
print(f"  Min correlation: {corr_flat.min():.3f}")

In [ ]:
# Covariance matrix heatmap
cov_matrix = Sigma  # Use the Ledoit-Wolf shrunk covariance matrix

plt.figure(figsize=(10, 8))
sns.heatmap(cov_matrix, annot=True, fmt='.4f', cmap='YlOrRd', 
            square=True, xticklabels=tickers, yticklabels=tickers)
plt.title('Covariance Matrix (Annualized) - Healthcare Sector', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\nKey Covariance Statistics:")
# Diagonal elements are variances
cov_array = cov_matrix.values  # Convert to numpy array
variances = np.diag(cov_array)
print(f"  Average variance: {variances.mean():.4f}")
print(f"  Max variance: {variances.max():.4f} ({tickers[np.argmax(variances)]})")
print(f"  Min variance: {variances.min():.4f} ({tickers[np.argmin(variances)]})")

# Off-diagonal covariances
cov_flat = cov_array[np.triu_indices_from(cov_array, k=1)]
print(f"\n  Average covariance: {cov_flat.mean():.4f}")
print(f"  Max covariance: {cov_flat.max():.4f}")
print(f"  Min covariance: {cov_flat.min():.4f}")

## Section 4: Mean-Variance Optimization

### Primer: Mean-Variance (Markowitz) Optimization

**Objective:** Maximize risk-adjusted returns by finding the optimal tradeoff between expected return and portfolio variance.

**Formula:**
$$\max_w \quad \mu^T w - \frac{\lambda}{2} w^T \Sigma w$$

Subject to:
- Long-only: $w_i \geq 0$
- Fully invested: $\sum w_i = 1$
- Max weight: $w_i \leq 20\%$

**Key Assumptions:**
- Returns are normally distributed
- Historical mean returns predict future returns
- Covariance structure remains stable

**When it works:**
- Stable market conditions
- Large, liquid stocks with history
- Mean-reverting returns

**When it fails:**
- Structural breaks or regime changes
- Small sample sizes (estimation error)
- Extreme events (fat tails)

In [ ]:
# Run Mean-Variance optimization
mv_results = mv_optimizer.optimize_mean_variance(
    mu=mu_hist.values,
    Sigma=Sigma.values,
    constraints_config=config['constraints'],
    rf=rf,
    lambda_grid=np.logspace(
        np.log10(config['mean_variance']['lambda_min']),
        np.log10(config['mean_variance']['lambda_max']),
        config['mean_variance']['lambda_points']
    ),
    w_before=before_weights
)

# Extract optimal weights (best Sharpe ratio)
w_mv = pd.Series(mv_results['weights'], index=tickers)

print("\n" + "="*70)
print("MEAN-VARIANCE OPTIMAL WEIGHTS")
print("="*70)
for ticker, weight in w_mv.items():
    print(f"{ticker:6s}: {weight:6.2%}")
print("="*70)

In [ ]:
# Calculate Mean-Variance metrics
mv_metrics = metrics.compute_portfolio_metrics(
    w_mv.values, mu_hist.values, Sigma.values, rf, before_weights, tickers
)

print("\nMean-Variance Portfolio Metrics:")
print(f"  Expected Annual Return:      {mv_metrics['expected_annual_return']:.2%}")
print(f"  Expected Annual Volatility:  {mv_metrics['expected_annual_volatility']:.2%}")
print(f"  Sharpe Ratio:                {mv_metrics['sharpe_ratio']:.3f}")
print(f"  Risk-Adjusted Return:        {mv_metrics['risk_adjusted_return']:.3f}")
print(f"  Diversification Ratio:       {mv_metrics['diversification_ratio']:.3f}")
print(f"  Effective # Assets:          {mv_metrics['effective_n_assets']:.2f}")
print(f"  Turnover vs Before:          {mv_metrics['turnover']:.2%}")

## Section 5: Black-Litterman with Sentiment Analysis (Pure Sentiment)

### Primer: Sentiment-Only Black-Litterman

**Objective:** Incorporate market sentiment from financial news into portfolio optimization without manual analyst views.

**Key Difference from Consumer Sector:**
- **100% Sentiment-Based:** No analyst views component
- Views derived purely from FinBERT sentiment analysis of recent news
- More objective, data-driven approach

**How it works:**
1. **Equilibrium Returns:** Calculate market-implied returns from current portfolio
2. **Sentiment Analysis:** Analyze financial news for each stock using FinBERT
3. **Generate Views:** Rank stocks by sentiment, assign expected returns
4. **Black-Litterman Formula:** Blend equilibrium with sentiment views
5. **Optimization:** Find optimal weights using posterior returns

**When it works:**
- News-driven markets
- High information flow period
- When market sentiment predicts returns

**When it struggles:**
- Low news volume
- Sentiment lags reality
- Contrarian opportunities (negative news = buying chance)

In [ ]:
# Calculate equilibrium returns (market-implied)
delta = config['black_litterman']['delta']
tau = config['black_litterman']['tau']

pi = bl_model.compute_equilibrium_returns(
    before_weights, Sigma.values, delta
)

print("Equilibrium Returns (market-implied):")
print("="*50)
for ticker, ret in zip(tickers, pi):
    print(f"{ticker:6s}: {ret:7.2%}")
print("="*50)

In [ ]:
# Run sentiment analysis for all Industrial stocks
# Load API keys from .env file
from dotenv import load_dotenv

load_dotenv()
polygon_key = os.getenv('POLYGON_API_KEY')
finnhub_key = os.getenv('FINNHUB_API_KEY')

print("Running sentiment analysis...")
print("This will take a few minutes due to API rate limiting.\n")

# Note: For Healthcare sector, we use pure sentiment (no analyst views)
sentiment_views, sentiment_data, sentiment_summary = sentiment_analyzer.analyze_portfolio_sentiment(
    tickers=tickers,
    config=config,
    polygon_key=polygon_key,
    finnhub_key=finnhub_key
)

print("\n" + "="*70)
print("SENTIMENT ANALYSIS COMPLETE")
print("="*70)

In [ ]:
# Display sentiment summary
print("\nSentiment Scores by Stock:")
print("="*70)

# Extract sentiment scores from sentiment_data
sentiment_scores = {ticker: data['sentiment_score'] for ticker, data in sentiment_data.items()}

for ticker in tickers:
    if ticker in sentiment_scores:
        score = sentiment_scores[ticker]
        print(f"{ticker:6s}: {score:+.3f}")
print("="*70)

# Show sentiment-based views (these will be used for Black-Litterman)
print("\nSentiment-Based Views for Black-Litterman:")
print("="*70)
for ticker, view in sentiment_views.items():
    print(f"{ticker:6s}: Expected return = {view['return']:+.2%}, "
          f"Confidence (Omega) = {view['confidence']:.6f}")
print("="*70)

In [ ]:
# Run Black-Litterman optimization with pure sentiment
bl_sentiment_results = bl_model.optimize_black_litterman_sentiment(
    pi=pi,
    Sigma=Sigma.values,
    combined_views=sentiment_views,  # Pure sentiment, no analyst views
    tickers=tickers,
    tau=tau,
    constraints_config=config['constraints'],
    risk_free_rate=rf,
    before_weights=before_weights,
    delta=delta
)

# Extract weights
w_bl_sentiment = pd.Series(bl_sentiment_results['weights'], index=tickers)
mu_BL_sentiment = bl_sentiment_results['posterior_returns']

print("\n" + "="*70)
print("BLACK-LITTERMAN (SENTIMENT) WEIGHTS")
print("="*70)
for ticker, weight in w_bl_sentiment.items():
    print(f"{ticker:6s}: {weight:6.2%}")
print("="*70)

In [ ]:
# Calculate BL-Sentiment metrics
bl_sentiment_metrics = metrics.compute_portfolio_metrics(
    w_bl_sentiment.values, mu_BL_sentiment, Sigma.values, rf, before_weights, tickers
)

print("\nBL-Sentiment Portfolio Metrics:")
print(f"  Expected Annual Return:      {bl_sentiment_metrics['expected_annual_return']:.2%}")
print(f"  Expected Annual Volatility:  {bl_sentiment_metrics['expected_annual_volatility']:.2%}")
print(f"  Sharpe Ratio:                {bl_sentiment_metrics['sharpe_ratio']:.3f}")
print(f"  Risk-Adjusted Return:        {bl_sentiment_metrics['risk_adjusted_return']:.3f}")
print(f"  Diversification Ratio:       {bl_sentiment_metrics['diversification_ratio']:.3f}")
print(f"  Effective # Assets:          {bl_sentiment_metrics['effective_n_assets']:.2f}")
print(f"  Turnover vs Before:          {bl_sentiment_metrics['turnover']:.2%}")

## Section 6: Results Comparison

In [ ]:
# Compare weights
weight_comparison = pd.DataFrame({
    'Before': before_weights,
    'Mean-Variance': w_mv.values,
    'BL (Sentiment)': w_bl_sentiment.values
}, index=tickers)

print("\n" + "="*90)
print("PORTFOLIO WEIGHTS COMPARISON")
print("="*90)
print(weight_comparison.to_string(float_format=lambda x: f'{x:.2%}'))
print("="*90)
print(f"Total: {weight_comparison.sum().to_string(float_format=lambda x: f'{x:.2%}')}")
print("="*90)

In [ ]:
# Plot weight comparison
fig, ax = plt.subplots(figsize=(12, 6))
weight_comparison.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Portfolio Weight Comparison - Healthcare Sector', fontsize=14, pad=20)
ax.set_xlabel('Ticker', fontsize=12)
ax.set_ylabel('Weight', fontsize=12)
ax.set_ylim(0, 0.25)
ax.axhline(y=0.20, color='red', linestyle='--', label='Max Weight (20%)')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare metrics
before_metrics = metrics.compute_portfolio_metrics(
    before_weights, mu_hist.values, Sigma.values, rf, None, tickers
)

metrics_comparison = pd.DataFrame({
    'Before': [
        before_metrics['expected_annual_return'],
        before_metrics['expected_annual_volatility'],
        before_metrics['sharpe_ratio'],
        before_metrics['diversification_ratio'],
        before_metrics['effective_n_assets'],
        0.0
    ],
    'Mean-Variance': [
        mv_metrics['expected_annual_return'],
        mv_metrics['expected_annual_volatility'],
        mv_metrics['sharpe_ratio'],
        mv_metrics['diversification_ratio'],
        mv_metrics['effective_n_assets'],
        mv_metrics['turnover']
    ],
    'BL (Sentiment)': [
        bl_sentiment_metrics['expected_annual_return'],
        bl_sentiment_metrics['expected_annual_volatility'],
        bl_sentiment_metrics['sharpe_ratio'],
        bl_sentiment_metrics['diversification_ratio'],
        bl_sentiment_metrics['effective_n_assets'],
        bl_sentiment_metrics['turnover']
    ]
}, index=[
    'Expected Annual Return',
    'Expected Annual Volatility',
    'Sharpe Ratio',
    'Diversification Ratio',
    'Effective # Assets',
    'Turnover'
])

print("\n" + "="*90)
print("PORTFOLIO METRICS COMPARISON")
print("="*90)
print(metrics_comparison.to_string(float_format=lambda x: f'{x:.4f}'))
print("="*90)

## Section 7: Export Results

In [ ]:
# Save weights to CSV
output_dir = config['output']['directory']
os.makedirs(output_dir, exist_ok=True)

weight_comparison.to_csv(f'{output_dir}/portfolio_weights.csv')
print(f"✓ Weights saved to {output_dir}/portfolio_weights.csv")

# Save metrics to CSV
metrics_comparison.to_csv(f'{output_dir}/portfolio_metrics.csv')
print(f"✓ Metrics saved to {output_dir}/portfolio_metrics.csv")

## Section 6: CVaR (Conditional Value-at-Risk) Optimization

### Primer: CVaR Optimization

**Objective:** Minimize Expected Shortfall (tail risk) rather than volatility.

**Formula:**
- Minimize: CVaR = alpha + (1/(1-beta)*T) * sum(z_t)
- Where alpha = Value-at-Risk threshold
- z_t = losses exceeding VaR in scenario t

**Key Assumptions:**
- Historical scenarios represent future tail events
- Focus on downside risk (asymmetric)
- Does not assume normal distribution

**When it works:**
- High tail risk / fat-tailed returns
- Risk-averse investors focused on downside
- Crisis periods or volatile markets

**When it struggles:**
- Low volatility / stable markets (similar to MV)
- Historical scenarios not representative of future

In [ ]:
# Load CVaR configuration
confidence = config['cvar']['confidence_level']

# Run CVaR optimization
cvar_results = cvar_optimizer.optimize_cvar(
    returns=returns,
    constraints_config=config['constraints'],
    confidence_level=confidence,
    w_before=before_weights
)

# Extract optimal weights
w_cvar = pd.Series(cvar_results['weights'], index=tickers)

print("\n" + "="*70)
print("CVaR OPTIMAL WEIGHTS")
print("="*70)
for ticker, weight in w_cvar.items():
    print(f"{ticker:6s}: {weight:6.2%}")
print("="*70)
print(f"\nCVaR ({confidence:.0%}): {cvar_results['cvar_value']:.6f} (daily loss)")
print(f"VaR ({confidence:.0%}):  {cvar_results['var_value']:.6f} (daily loss)")
print(f"Expected Return: {cvar_results['expected_return']:.2%} (annualized)")
print(f"Volatility: {cvar_results['volatility']:.2%} (annualized)")

In [ ]:
# Calculate CVaR portfolio metrics
cvar_metrics = metrics.compute_portfolio_metrics(
    w_cvar.values, mu_hist.values, Sigma.values, rf, before_weights, tickers
)

print("\nCVaR Portfolio Metrics:")
print(f"  Expected Annual Return:      {cvar_metrics['expected_annual_return']:.2%}")
print(f"  Expected Annual Volatility:  {cvar_metrics['expected_annual_volatility']:.2%}")
print(f"  Sharpe Ratio:                {cvar_metrics['sharpe_ratio']:.3f}")
print(f"  Risk-Adjusted Return:        {cvar_metrics['risk_adjusted_return']:.3f}")
print(f"  Diversification Ratio:       {cvar_metrics['diversification_ratio']:.3f}")
print(f"  Effective # Assets:          {cvar_metrics['effective_n_assets']:.2f}")
print(f"  Turnover vs Before:          {cvar_metrics['turnover']:.2%}")

## Section 8: Historical Backtest

Test how these portfolios would have actually performed over the historical period.

In [ ]:
# Backtest Before portfolio
before_backtest = backtester.backtest_portfolio(
    weights=before_weights,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Before portfolio backtested')

In [ ]:
# Backtest Mean-Variance portfolio
mv_backtest = backtester.backtest_portfolio(
    weights=w_mv,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Mean-Variance portfolio backtested')

In [ ]:
# Backtest Black-Litterman (Sentiment) portfolio
bl_sentiment_backtest = backtester.backtest_portfolio(
    weights=w_bl_sentiment,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Black-Litterman (Sentiment) portfolio backtested')

In [ ]:
# Backtest CVaR portfolio
cvar_backtest = backtester.backtest_portfolio(
    weights=w_cvar,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ CVaR portfolio backtested')

In [ ]:
# Combine backtest results and display
backtest_results = {
    'Before': before_backtest,
    'Mean-Variance': mv_backtest,
    'BL (Sentiment)': bl_sentiment_backtest,
    'CVaR': cvar_backtest
}

# Plot cumulative returns and display metrics
backtester.summarize_and_plot(backtest_results, rf_annual=rf)

---

## Summary

### Healthcare Sector Optimization Complete

**Stocks Analyzed:** ABBV, JAZZ, THC, MRK, UNH, VEEV, AZN

**Methods Implemented:**
1. Mean-Variance Optimization
2. Black-Litterman with Pure Sentiment Analysis

**Key Differences from Consumer Sector:**
- No analyst views (pure sentiment approach)
- Different risk-return profile
- Healthcare sector specific dynamics

**Files Generated:**
- `outputs_healthcare/portfolio_weights.csv`
- `outputs_healthcare/portfolio_metrics.csv`
- Correlation heatmap
- Weight comparison chart
- Backtest cumulative returns plot